In [0]:
# IMPORTS
import logging
from pyspark.sql import DataFrame, SparkSession
from delta.tables import DeltaTable
from pyspark.sql.functions import col
from pyspark.sql.utils import AnalysisException
from databricks.sdk.runtime import *

In [0]:
# Import the module
import sys
sys.path.append("../aws_utils")
import aws_utils
# Get Environment Prefix
env = aws_utils.get_env_prefix()

In [0]:
from datetime import datetime

# Get max BusinessDate from both tables
dunderlaw_max = spark.sql(f"""
    SELECT MAX(BusinessDate) as max_dt
    FROM `{env}-cdh`.derivative.dunderlaw
""").collect()[0]["max_dt"]

party_max = spark.sql(f"""
    SELECT MAX(BusinessDate) as max_dt
    FROM `{env}-cdh`.derivative.dparty
""").collect()[0]["max_dt"]

print("dunderlaw max:", dunderlaw_max)
print("party max:", party_max)

# Check equality
if dunderlaw_max != party_max:
    raise Exception(
        "ERROR: Latest BusinessDate of derivative.dunderlaw "
        "and derivative.dparty are not equal. "
        "Probably the main flow has not finished yet."
    )

# If equal, set Run_Date
Run_Date = dunderlaw_max  # already date/timestamp from Spark
print("Run_Date:", Run_Date)

In [0]:
# calculation of LawPerimeter
query1 = f"""WITH POOL_OF_MAXBD_LOWER_THAN_RUNDT AS
(
  SELECT distinct
        P.PortfolioName,
        P.CDI,
        P.TaxRegistrationNumber  as TaxNumber,
        MAX(UL.BusinessDate) MaxBusinessDate
  FROM `{env}-cdh`.derivative.dunderlaw            UL
  INNER JOIN `{env}-cdh`.derivative.dparty         P     ON P.CDI = UL.CDI AND P.BusinessDate = '{Run_Date}'
  WHERE 1=1
  AND UL.underlawtype = 'N3869' 
  AND P.PortfolioName in ('Mercury', 'Orbit', 'Light', 'Cell', 'Pearl Portfolio A') 
  AND P.TaxRegistrationNumber IN ('059606538','069235687','137620015')
  GROUP BY P.PortfolioName, P.CDI, P.TaxRegistrationNumber, UL.SubmissionDate
  HAVING max(UL.BusinessDate) < '{Run_Date}'
)
/*Returs the rows that have max BusinessDate previous to Run_Date*/
SELECT distinct
       MAXBD.PortfolioName,
       MAXBD.CDI,
       MAXBD.TaxNumber,
       UL.SubmissionDate
FROM `{env}-cdh`.derivative.dunderlaw            UL
INNER JOIN POOL_OF_MAXBD_LOWER_THAN_RUNDT    MAXBD ON  MAXBD.CDI        = UL.CDI 
                                                   AND UL.BusinessDate  = MAXBD.MaxBusinessDate 
                                                   AND UL.PortfolioName = MAXBD.PortfolioName
WHERE 1=1
AND UL.underlawtype = 'N3869' 

UNION

/*Returs the rows that have BusinessDate for the Run_Date*/
SELECT distinct
       P.PortfolioName,
       P.CDI,
       P.TaxRegistrationNumber  as TaxNumber,
       UL.SubmissionDate
FROM `{env}-cdh`.derivative.dunderlaw            UL
INNER JOIN `{env}-cdh`.derivative.dparty         P     ON P.CDI = UL.CDI AND P.BusinessDate = '{Run_Date}'
WHERE 1=1
AND UL.underlawtype = 'N3869' 
AND UL.BusinessDate = '{Run_Date}'
AND P.PortfolioName in ('Mercury', 'Orbit', 'Light', 'Cell', 'Pearl Portfolio A')"""

df_LawPerimeter = spark.sql(query1)
df_LawPerimeter.createOrReplaceTempView("LawPerimeter")
# display(df_LawPerimeter)


# calculation of Master_Index
query2 = f""" select distinct
                     a.VATNumber,
                     a.L3869Status,
                     a.Pool,
                     CAST(REPLACE(a.ExposureToAllCreditors, ',', '.') AS DOUBLE) as ExposureToAllCreditors,
                     CAST(REPLACE(a.ApplicationExposureHoist, ',', '.') AS DOUBLE) as ApplicationExposureHoist,
                     CAST(REPLACE(a.TotalRepaymentAmount, ',', '.') AS DOUBLE) as TotalRepaymentAmount,
                     CAST(REPLACE(a.TotalArticles, ',', '.') AS DOUBLE) as TotalArticles,
                     a.BusinessDate as ExtBusinessDateLoading,
                     a.IngestionTimestamp as ExtIngestionTimestamp, 
                     b.SubmissionDate
from `{env}-cdh`.externalfiles.exthoist_master_index_accepted a 
left join LawPerimeter b on a.VATNumber = b.TaxNumber
where a.IngestionTimestamp = (select max(IngestionTimestamp) from `{env}-cdh`.externalfiles.exthoist_master_index_accepted)"""

df_Master_Index = spark.sql(query2)
df_Master_Index.createOrReplaceTempView("Master_Index")
# display(df_Master_Index)


# calculation of First_Data
query3 = f""" with First_Data_v0 as 
(
  select PortfolioName
        ,date_add(month,2,MIN(AssignmentDate)) as TwoMontsAfterAssignmentDate 
  from `{env}-cdh`.primary.portfolio 
  where AssetOwner = 'Hoist Finance AB (PUBL)'
  group by PortfolioName
)
select distinct fd.PortfolioName, c.currentmonthlastworkingdate as MinDate 
from First_Data_v0 fd
inner join `{env}-cdh`.reference.ref_calendar c on month(c.currentmonthlastworkingdate) = month(fd.TwoMontsAfterAssignmentDate) and year(c.currentmonthlastworkingdate) = year(fd.TwoMontsAfterAssignmentDate)
order by 1
"""

df_First_Data = spark.sql(query3)
df_First_Data.createOrReplaceTempView("First_Data")
# display(df_First_Data)

In [0]:
# --------------------------------------------------------- Calculation of SubmissionsBeforeMigration --------------------------------------------------------- (has 6 steps: SubmissionsBeforeMigration_v0, SubmissionsBeforeMigration_v1, MinDateIAANotNULL, SubmissionsBeforeMigration_v2, SubmissionsBeforeMigration_v3, SubmissionsBeforeMigration)

# SubmissionsBeforeMigration_v0
query4 = f"""select distinct a.*, b.Mindate, b.PortfolioName
from Master_Index a
left join First_Data b ON a.SubmissionDate <= b.MinDate
order by SubmissionDate desc
"""

df_SubmissionsBeforeMigration_v0 = spark.sql(query4)
df_SubmissionsBeforeMigration_v0.createOrReplaceTempView(
    "SubmissionsBeforeMigration_v0"
)
# display(df_SubmissionsBeforeMigration_v0)


# calculation of SubmissionsBeforeMigration_v1
query5 = f""" select distinct a.VATNumber, a.L3869Status, a.Pool, a.ExposureToAllCreditors, a.ApplicationExposureHoist, a.TotalRepaymentAmount, a.TotalArticles, a.SubmissionDate, a.MinDate, a.PortfolioName, b.ExposureKey as AccountNumber, a.ExtBusinessDateLoading, a.ExtIngestionTimestamp
from SubmissionsBeforeMigration_v0 a
left join `{env}-cdh`.derivative.dexposure b on b.PartyTaxRegistrationNumber = a.VATNumber and b.PortfolioName = a.PortfolioName  
order by VATNumber, PortfolioName
"""

df_SubmissionsBeforeMigration_v1 = spark.sql(query5)
df_SubmissionsBeforeMigration_v1.createOrReplaceTempView(
    "SubmissionsBeforeMigration_v1"
)
# display(df_SubmissionsBeforeMigration_v1)


# calculation of MinDateIAANotNULL
query6 = f"""select e.PortfolioName, e.ExposureKey as AccountNumber, MIN(e.BusinessDate) MinDateIAANotNULL
from `{env}-cdh`.derivative.dexposure e
inner join `{env}-cdh`.primary.portfolio p on p.PortfolioName = e.PortfolioName
where p.AssetOwner = 'Hoist Finance AB (PUBL)'
and InitialAssignmentAmount is not null
and InitialAssignmentAmount > 0
group by e.PortfolioName, e.ExposureKey
order by 1,2
"""

df_MinDateIAANotNULL = spark.sql(query6)
df_MinDateIAANotNULL.createOrReplaceTempView("MinDateIAANotNULL")
# display(df_MinDateIAANotNULL)


# calculation of SubmissionsBeforeMigration_v2
query7 = f"""select a.*,b.MinDateIAANotNULL, c.InitialAssignmentAmount
from SubmissionsBeforeMigration_v1 a
left join MinDateIAANotNULL b on a.AccountNumber = b.AccountNumber and a.PortfolioName = b.PortfolioName
left join `{env}-cdh`.derivative.dexposure c on c.ExposureKey = a.AccountNumber and c.BusinessDate = b.MinDateIAANotNULL
"""

df_SubmissionsBeforeMigration_v2 = spark.sql(query7)
df_SubmissionsBeforeMigration_v2.createOrReplaceTempView(
    "SubmissionsBeforeMigration_v2"
)
# display(df_SubmissionsBeforeMigration_v2)


# calculation of SubmissionsBeforeMigration_v3
query8 = f""" with SubmissionsBeforeMigration_v3_pre_part as 
(
 select DISTINCT a.*
 	          ,CASE WHEN a.Pool like '%Pool%' AND upper(a.PortfolioName) != upper('Mercury') THEN NULL ELSE InitialAssignmentAmount END AS TLC
 from SubmissionsBeforeMigration_v2 a
 where a.MinDate is not null
)
select VATNumber, L3869Status, Pool, ExposureToAllCreditors, ApplicationExposureHoist, TotalRepaymentAmount, TotalArticles, SubmissionDate, MinDate, PortfolioName, SUM(TLC) TLC, ExtBusinessDateLoading, ExtIngestionTimestamp
from SubmissionsBeforeMigration_v3_pre_part
group by VATNumber, L3869Status, Pool, ExposureToAllCreditors, ApplicationExposureHoist, TotalRepaymentAmount, TotalArticles, SubmissionDate, MinDate, PortfolioName, ExtBusinessDateLoading, ExtIngestionTimestamp
"""

df_SubmissionsBeforeMigration_v3 = spark.sql(query8)
df_SubmissionsBeforeMigration_v3.createOrReplaceTempView(
    "SubmissionsBeforeMigration_v3"
)
# display(df_SubmissionsBeforeMigration_v3)


# calculation of SubmissionsBeforeMigration
query9 = f""" select VATNumber, L3869Status, Pool, ExposureToAllCreditors, ApplicationExposureHoist, TotalRepaymentAmount, CAST(REPLACE(TotalArticles,',', '') AS DOUBLE) TotalArticles, SubmissionDate, CAST(sum(TLC) AS FLOAT) TLC, ExtBusinessDateLoading, ExtIngestionTimestamp
from SubmissionsBeforeMigration_v3 
WHERE TLC IS NOT NULL
group by VATNumber, L3869Status, Pool, ExposureToAllCreditors, ApplicationExposureHoist, TotalRepaymentAmount, CAST(REPLACE(TotalArticles,',', '') AS DOUBLE), SubmissionDate, ExtBusinessDateLoading, ExtIngestionTimestamp
"""

df_SubmissionsBeforeMigration = spark.sql(query9)
df_SubmissionsBeforeMigration.createOrReplaceTempView("SubmissionsBeforeMigration")
# display(df_SubmissionsBeforeMigration)

In [0]:
# ------------  SUBMISSIONS AFTER MIGRATION  ------------ (HAS 3 STEPS: SubmissionsAfterMigration_v0, SubmissionsAfterMigration_v1, SubmissionsAfterMigration)

# calculation of SubmissionsAfterMigration_v0
query10 = f""" SELECT distinct a.*, MONTH(DATEADD(MONTH, -1, SubmissionDate)) MinDateMonth, YEAR(DATEADD(MONTH, -1, SubmissionDate)) MinDateYear, b.PortfolioName
FROM Master_Index a
LEFT JOIN First_Data b ON a.SubmissionDate > b.MinDate
where MinDate is not null
order by SubmissionDate desc
"""

df_SubmissionsAfterMigration_v0 = spark.sql(query10)
df_SubmissionsAfterMigration_v0.createOrReplaceTempView("SubmissionsAfterMigration_v0")
# display(df_SubmissionsAfterMigration_v0)


# calculation of SubmissionsAfterMigration_v1
query11 = f""" with SubmissionsAfterMigration_v1_pre_part as 
( 
 select DISTINCT a.* 
 	 , CASE WHEN Pool like '%Pool%' AND upper(a.PortfolioName) != upper('Mercury') THEN NULL ELSE c.OutstandingTotalBalance END AS TLC
 from SubmissionsAfterMigration_v0 a
 left join `{env}-cdh`.derivative.dexposure b on b.PartyTaxRegistrationNumber = a.VATNumber and b.PortfolioName = a.PortfolioName
 left join `{env}-cdh`.derivative.dexposure c on c.PartyTaxRegistrationNumber = a.VATNumber and month(c.BusinessDate) = a.MinDateMonth and year(c.BusinessDate) = a.MinDateYear  and c.PortfolioName = a.PortfolioName
)
select VATNumber, L3869Status, Pool, ExposureToAllCreditors, ApplicationExposureHoist, TotalRepaymentAmount, TotalArticles, SubmissionDate, MinDateMonth, MinDateYear, PortfolioName, SUM(TLC) TLC, ExtBusinessDateLoading, ExtIngestionTimestamp
from SubmissionsAfterMigration_v1_pre_part 
group by VATNumber, L3869Status, Pool, ExposureToAllCreditors, ApplicationExposureHoist, TotalRepaymentAmount, TotalArticles, SubmissionDate, MinDateMonth, MinDateYear, PortfolioName, ExtBusinessDateLoading, ExtIngestionTimestamp"""

df_SubmissionsAfterMigration_v1 = spark.sql(query11)
df_SubmissionsAfterMigration_v1.createOrReplaceTempView("SubmissionsAfterMigration_v1")
# display(df_SubmissionsAfterMigration_v1)


# calculation of SubmissionsAfterMigration
query12 = f""" SELECT VATNumber, L3869Status, Pool, ExposureToAllCreditors, ApplicationExposureHoist, TotalRepaymentAmount, CAST(REPLACE(TotalArticles,',', '') AS DOUBLE) TotalArticles, SubmissionDate, CAST(sum(TLC) AS DOUBLE) TLC, ExtBusinessDateLoading, ExtIngestionTimestamp
FROM SubmissionsAfterMigration_v1
where TLC is not null
GROUP BY VATNumber, L3869Status, Pool, ExposureToAllCreditors, ApplicationExposureHoist, TotalRepaymentAmount, CAST(REPLACE(TotalArticles,',', '') AS DOUBLE), SubmissionDate, ExtBusinessDateLoading, ExtIngestionTimestamp
"""

df_SubmissionsAfterMigration = spark.sql(query12)
df_SubmissionsAfterMigration.createOrReplaceTempView("SubmissionsAfterMigration")
# display(df_SubmissionsAfterMigration)

In [0]:
# **************************************************************Union before & after migration********************************************************************

# Final_v0 - sum TLC group by VATNumber to account for scenario where a VATNumber has both a pre-migration and post-migration TLC
query13 = f""" SELECT VATNumber, L3869Status, Pool, ExposureToAllCreditors, ApplicationExposureHoist, TotalRepaymentAmount, TotalArticles, SubmissionDate,
    COALESCE(SUM(CASE WHEN Source = 'PRE' THEN TLC END), 0) AS TLCBeforeMigration,
    COALESCE(SUM(CASE WHEN Source = 'POST' THEN TLC END), 0) AS TLCAfterMigration,
    SUM(TLC) AS TLC,
    ExtBusinessDateLoading, ExtIngestionTimestamp
FROM (
	SELECT *, 'PRE' AS Source
	FROM SubmissionsBeforeMigration

	UNION ALL

	SELECT *, 'POST' AS Source
	FROM SubmissionsAfterMigration
) A
GROUP BY VATNumber, L3869Status, Pool, ExposureToAllCreditors, ApplicationExposureHoist, TotalRepaymentAmount, SubmissionDate, TotalArticles, ExtBusinessDateLoading, ExtIngestionTimestamp
"""

df_Final_v0 = spark.sql(query13)
df_Final_v0.createOrReplaceTempView("Final_v0")
# display(df_Final_v0)


# calculation of Final_df
query14 = f""" SELECT *
	 , (TLC - TotalArticles) / TLC AS Haircut --Calculation of Haircut
    , current_timestamp() AS IngestionTimestamp
FROM Final_v0

UNION

SELECT 
    VATNumber, 
    L3869Status, 
    Pool, 
    cast(REPLACE(ExposureToAllCreditors,',', '.') as double) ExposureToAllCreditors, 
    cast(REPLACE(ApplicationExposureHoist,',','.') as double) ApplicationExposureHoist, 
    cast(REPLACE(TotalRepaymentAmount,',','.') as double) TotalRepaymentAmount, 
    cast(REPLACE(TotalArticles,',', '.') AS DOUBLE) TotalArticles,
    null as SubmissionDate, 
    null as TLCBeforeMigration, 
    null as TLCAfterMigration, 
    null as TLC, 
    BusinessDate as ExtBusinessDateLoading, 
    Ingestiontimestamp as ExtIngestionTimestamp,
    null as Haircut,  
    current_timestamp() AS IngestionTimestamp
FROM `{env}-cdh`.externalfiles.exthoist_master_index_accepted
WHERE 1 = 1
AND VATNUMBER NOT IN (SELECT DISTINCT VATNUMBER FROM Final_v0)
AND IngestionTimestamp = (select max(IngestionTimestamp) from `{env}-cdh`.externalfiles.exthoist_master_index_accepted)

"""

Final_df = spark.sql(query14)
Final_df.createOrReplaceTempView("Final_df")
# display(Final_df)


# insert into final target tbl Hoist_3869AcceptedHaircut
Final_df.write.mode("append").saveAsTable(
    f"`{env}-cdh-delivery`.hoist.Hoist_3869AcceptedHaircut"
)